# PelotonIQ — LangGraph Agent

Multi-agent system that combines structured data lookup, hybrid search,
ML predictions, and race commentary into a unified pre-race intelligence platform.

## Architecture
```
User query
    │
    ▼
router_node        — classifies query, extracts params
    │
    ├── structured_node   — dataframe tool calls (who won X?)
    ├── retriever_node    — hybrid search over course/rider embeddings
    │       │
    │       ├── commentary_node  — adds race commentary context
    │       └── predictor_node   — adds ML probability distributions
    │
    └── synthesizer_node  — calls Claude to generate final response
```

## Query types
- `STRUCTURED` — specific fact lookup via dataframe (who won X, stage results)
- `SEMANTIC_COURSE` — terrain/course profile questions
- `SEMANTIC_RIDER` — rider performance/season questions
- `PREDICTIVE` — pre-race analysis requiring ML model
- `HYBRID` — requires both course context and rider context


In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import re
import json
import pickle
from pathlib import Path
from typing import TypedDict, Annotated, Optional
from dotenv import load_dotenv

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue, Range
from rank_bm25 import BM25Okapi
import anthropic

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

load_dotenv()

PROCESSED_DATA  = Path('../data/processed')
MODELS_DIR      = Path('../models')
COMMENTARY_DIR  = Path('../data/commentary')
QDRANT_URL      = 'http://localhost:6333'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

embed_model = SentenceTransformer(EMBEDDING_MODEL)
qdrant      = QdrantClient(url=QDRANT_URL)
claude      = anthropic.Anthropic()

c:\Users\Admin\Desktop\GitHubPortfolio\peloton-iq\.venv\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
print(f'Collections: {[c.name for c in qdrant.get_collections().collections]}')

Collections: ['course_profiles', 'rider_seasons']


In [3]:
# Load all data sources
merged_df   = pd.read_csv(PROCESSED_DATA / "merged_uci_races.csv", low_memory=False)
course_data = pd.read_csv(PROCESSED_DATA / "course_data_clean.csv")
model_df    = pd.read_csv(PROCESSED_DATA / "model_df.csv", low_memory=False)  # add this

merged_df["Date"] = pd.to_datetime(merged_df["Date"])

# Load prediction model
with open(MODELS_DIR / "tier_predictor.pkl", "rb") as f:
    predictor = pickle.load(f)

pred_model      = predictor["model"]
pred_model_name = predictor["model_name"]
FEATURE_COLS    = predictor["feature_cols"]
TIER_ORDER      = predictor["tier_order"]
TIER_TO_INT     = predictor["tier_to_int"]
xgb_medians     = pd.Series(predictor["xgb_medians"])
needs_imp       = pred_model_name == "XGBoost"

print(f"merged_df:    {merged_df.shape[0]:,} rows")
print(f"course_data:  {course_data.shape[0]:,} rows")
print(f"model_df:     {model_df.shape[0]:,} rows  ← has rider history features")
print(f"Pred model:   {pred_model_name}")
print(f"Top-5 acc:    {predictor['metrics'][pred_model_name]['top5']*100:.1f}%")

merged_df:    140,573 rows
course_data:  963 rows
model_df:     140,573 rows  ← has rider history features
Pred model:   XGBoost
Top-5 acc:    55.2%


In [4]:
# Rebuild BM25 corpora — needed for hybrid search

def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

def serialize_course_doc(row: pd.Series) -> str:
    parts = [f"Race: {row['Race Name']}."]
    if pd.notna(row.get('Distance')):
        parts.append(f"Distance: {row['Distance']:.1f} km.")
    if pd.notna(row.get('Vertical Gain')) and pd.notna(row.get('Highest Elevation')):
        vg = row['Vertical Gain']
        if vg > 4000:   stage_type = 'a high mountain stage'
        elif vg > 2000: stage_type = 'a hilly stage with significant climbing'
        elif vg > 800:  stage_type = 'a moderately hilly stage'
        else:           stage_type = 'a flat stage suited to sprinters'
        parts.append(
            f"This is {stage_type} with {vg:.0f}m of vertical gain, "
            f"reaching a maximum elevation of {row['Highest Elevation']:.0f}m "
            f"and a minimum elevation of {row.get('Lowest Elevation', 0):.0f}m."
        )
    surfaces = {
        'Asphalt': 'asphalt', 'Road': 'road', 'Cobblestones': 'cobblestones',
        'Compacted Gravel': 'compacted gravel', 'Unpaved': 'unpaved sections',
    }
    surface_parts = []
    for col, label in surfaces.items():
        val = row.get(col)
        if pd.notna(val) and float(val) > 0.5:
            surface_parts.append(f'{val:.1f}km of {label}')
    if surface_parts:
        parts.append(f"Surface: {', '.join(surface_parts)}.")
    cob = row.get('Cobblestones')
    if pd.notna(cob) and float(cob) > 0:
        parts.append(f"Contains {cob:.1f}km of cobblestones.")
    return ' '.join(parts)

def serialize_rider_doc(rider_name: str, year: int, df: pd.DataFrame) -> str:
    rider_df = df[(df['Name'] == rider_name) & (df['Year_results'] == year)].copy()
    if rider_df.empty: return ''
    parts = [f"Rider: {rider_name}. Season: {year}."]
    team = rider_df['Team'].iloc[0]
    if pd.notna(team): parts.append(f"Team: {team}.")
    wins    = rider_df[rider_df['Rank'] == 1]
    top10s  = rider_df[rider_df['Top10'] == 1]
    dnfs    = rider_df[~rider_df['Did_Finish']]
    parts.append(
        f"Results: {len(wins)} wins, {len(top10s)} top-10 finishes, "
        f"{len(dnfs)} DNFs across {rider_df['Race_results'].nunique()} races."
    )
    if not wins.empty:
        parts.append(f"Won: {', '.join(wins['Race Name'].tolist()[:5])}.")
    gc_races = ['Tour de France', 'Giro d', 'Vuelta a Espana']
    for gc in gc_races:
        gc_res = rider_df[rider_df['Race_results'].str.contains(gc, na=False)]
        if not gc_res.empty:
            best = gc_res[gc_res['Did_Finish']]['Rank'].min()
            if best < 999: parts.append(f"{gc}: best result P{int(best)}.")
    return ' '.join(parts)

# Build corpora
course_corpus, course_ids = [], []
for _, row in course_data.iterrows():
    text = serialize_course_doc(row)
    if text.strip():
        course_corpus.append(text)
        course_ids.append(row['Race Name'])

rider_corpus, rider_ids = [], []
rider_seasons = merged_df.groupby(['Name', 'Year_results']).size().reset_index()
for _, row in rider_seasons.iterrows():
    text = serialize_rider_doc(row['Name'], row['Year_results'], merged_df)
    if text.strip():
        rider_corpus.append(text)
        rider_ids.append(f"{row['Name']}_{int(row['Year_results'])}")

bm25_course = BM25Okapi([tokenize(d) for d in course_corpus])
bm25_rider  = BM25Okapi([tokenize(d) for d in rider_corpus])

print(f'Course corpus: {len(course_corpus):,} docs')
print(f'Rider corpus:  {len(rider_corpus):,} docs')


Course corpus: 963 docs
Rider corpus:  5,835 docs


In [5]:
# ── Agent state — flows through every node ───────────────

class PelotonState(TypedDict):
    # Input
    query:               str

    # Routing
    query_type:          str        # STRUCTURED|SEMANTIC_COURSE|SEMANTIC_RIDER|PREDICTIVE|HYBRID
    structured_params:   dict       # extracted race/rider/year/stage params

    # Retrieved context — each node adds to these
    structured_data:     dict       # from dataframe tool calls
    course_context:      str        # from hybrid search over course profiles
    rider_context:       str        # from hybrid search over rider seasons
    commentary_context:  str        # from commentary layer
    prediction_context:  str        # from ML model — formatted as text

    # Output
    final_response:      str
    error:               str
    steps_taken:         list


def empty_state(query: str) -> PelotonState:
    """Initialize a clean state for a new query."""
    return {
        'query':              query,
        'query_type':         '',
        'structured_params':  {},
        'structured_data':    {},
        'course_context':     '',
        'rider_context':      '',
        'commentary_context': '',
        'prediction_context': '',
        'final_response':     '',
        'error':              '',
        'steps_taken':        [],
    }

In [51]:
# ── Dataframe tool functions ──────────────────────────────
# These are called by structured_node directly
# No embeddings — pure dataframe operations

def get_stage_winner(race_name: str, year: int, stage: int = None) -> dict:
    df = merged_df.copy()
    df = df[df['Year_results'] == year]
    df = df[df['Race_results'].str.contains(race_name, na=False, case=False)]
    if stage is not None:
        df = df[
            (df['Stage_results'] == stage) |
            (df['Stage_results'] == float(stage))
        ]
    if df.empty:
        return {'error': f'No results found for {race_name} {year}'}
    winner = df[df['Rank'] == 1]
    if winner.empty:
        return {'error': f'No rank 1 found for {race_name} {year}'}
    row = winner.iloc[0]
    return {
        'race':   row['Race Name'],
        'winner': row['Name'],
        'team':   row['Team'],
        'year':   year,
    }


def get_race_results(race_name: str, year: int, top_n: int = 10) -> dict:
    df = merged_df[
        merged_df['Race_results'].str.contains(race_name, na=False, case=False) &
        (merged_df['Year_results'] == year) &
        merged_df['Did_Finish']
    ].copy()
    if df.empty:
        return {'error': f'No results found for {race_name} {year}'}
    one_day = df['Stage_results'].isna().all()
    if one_day:
        top = df.nsmallest(top_n, 'Rank')[['Rank', 'Name', 'Team', 'Race Name']]
    else:
        top = (
            df.groupby('Name')['Rank'].min().reset_index()
            .nsmallest(top_n, 'Rank')
            .merge(df[['Name', 'Team']].drop_duplicates('Name'), on='Name')
        )
    return {'race': race_name, 'year': year, 'results': top.to_dict(orient='records')}


def get_rider_results(rider_name: str, year: int) -> dict:
    df = merged_df[
        merged_df['Name'].str.contains(rider_name, na=False, case=False) &
        (merged_df['Year_results'] == year)
    ].copy()
    if df.empty:
        return {'error': f'No results found for {rider_name} in {year}'}
    return {
        'rider':           df['Name'].iloc[0],
        'team':            df['Team'].iloc[0],
        'year':            year,
        'races_competed':  df['Race_results'].nunique(),
        'wins':            df[df['Rank'] == 1]['Race Name'].tolist(),
        'podiums':         df[df['Top3'] == 1]['Race Name'].tolist(),
        'top10s':          int(df['Top10'].sum()),
        'dnfs':            int((~df['Did_Finish']).sum()),
    }


def get_stage_profile(race_name: str, year: int, stage: int = None) -> dict:
    df = course_data.copy()
    if stage is not None:
        pattern = f'{year} {race_name} Stage {stage}'
        df = df[df['Race Name'] == pattern]
    else:
        df = df[
            df['Race Name'].str.contains(race_name, na=False, case=False) &
            (df['Year'] == year)
        ]
    if df.empty:
        return {'error': f'No course data found for {race_name} {year}'}
    row = df.iloc[0]
    return {
        'race':              row['Race Name'],
        'distance_km':       row.get('Distance'),
        'vertical_gain_m':   row.get('Vertical Gain'),
        'highest_elev_m':    row.get('Highest Elevation'),
        'lowest_elev_m':     row.get('Lowest Elevation'),
        'cobblestones_km':   row.get('Cobblestones'),
        'net_gain_m':        row.get('Net Gain'),
        'downhill_m':        row.get('Downhill'),
    }

def get_best_mountain_riders(
    min_vertical_gain: float = 4000,
    top_n: int = 10
) -> dict:
    """
    Find riders with best historical performance on mountain stages
    using direct dataframe computation — more reliable than retrieval
    for numeric terrain queries.
    """
    # Get course data for qualifying stages
    mountain_stages = course_data[
        course_data['Vertical Gain'] >= min_vertical_gain
    ]['Race Name'].tolist()

    if not mountain_stages:
        return {'error': f'No stages found with >{min_vertical_gain}m vertical gain'}

    # Filter race results to those stages
    mountain_results = merged_df[
        merged_df['Race Name'].isin(mountain_stages) &
        merged_df['Did_Finish']
    ].copy()

    if mountain_results.empty:
        return {'error': 'No results found for mountain stages'}

    # Compute per-rider stats on mountain stages
    rider_stats = (
        mountain_results
        .groupby('Name')
        .agg(
            races        = ('Race Name', 'count'),
            wins         = ('Rank', lambda x: (x == 1).sum()),
            top3         = ('Top3', 'sum'),
            top10        = ('Top10', 'sum'),
            avg_rank     = ('Rank', 'mean'),
            team         = ('Team', 'last'),
        )
        .reset_index()
    )

    # Require minimum race count for meaningful stats
    rider_stats = rider_stats[rider_stats['races'] >= 3]
    rider_stats['top10_rate'] = rider_stats['top10'] / rider_stats['races']
    rider_stats['win_rate']   = rider_stats['wins']  / rider_stats['races']

    # Sort by top10 rate then avg rank
    top_riders = (
        rider_stats
        .sort_values(['top10_rate', 'avg_rank'], ascending=[False, True])
        .head(top_n)
    )

    return {
        'mountain_stage_count': len(mountain_stages),
        'min_vertical_gain':    min_vertical_gain,
        'top_riders': top_riders[[
            'Name', 'team', 'races', 'wins',
            'top10', 'top10_rate', 'avg_rank'
        ]].to_dict(orient='records')
    }


def dispatch_tool(params: dict) -> dict:
    """Route structured_params to the right tool function."""
    fn = params.get('function', '')
    if fn == 'get_stage_winner':
        return get_stage_winner(
            params.get('race_name', ''),
            params.get('year', 2023),
            params.get('stage')
        )
    elif fn == 'get_race_results':
        return get_race_results(
            params.get('race_name', ''),
            params.get('year', 2023),
            params.get('top_n', 10)
        )
    elif fn == 'get_rider_results':
        return get_rider_results(
            params.get('rider_name', ''),
            params.get('year', 2023)
        )
    elif fn == 'get_stage_profile':
        return get_stage_profile(
            params.get('race_name', ''),
            params.get('year', 2023),
            params.get('stage')
        )
    elif fn == 'get_best_mountain_riders':
        return get_best_mountain_riders(
            min_vertical_gain=params.get('min_vertical_gain', 4000),
            top_n=params.get('top_n', 10)
        )
    return {'error': f'Unknown function: {fn}'}

# Quick verification
print(json.dumps(get_stage_winner('Tour de France', 2023, stage=17), indent=2, default=str))

{
  "race": "2023 Tour de France Stage 17",
  "winner": "GALL Felix",
  "team": "AG2R Citro\u00ebn Team",
  "year": 2023
}


In [52]:
# ── Hybrid search functions ───────────────────────────────

def semantic_search(query: str, collection: str, top_k: int = 20) -> list[dict]:
    qv = embed_model.encode(query).tolist()
    results = qdrant.query_points(
        collection_name=collection,
        query=qv, limit=top_k, with_payload=True
    )
    return [
        {'id': r.payload.get('doc_id'), 'score': r.score,
         'text': r.payload.get('text', '')}
        for r in results.points
    ]


def bm25_search(
    query: str, corpus: list, ids: list,
    bm25_index: BM25Okapi, top_k: int = 20
) -> list[dict]:
    scores  = bm25_index.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [
        {'id': ids[i], 'score': float(scores[i]), 'text': corpus[i]}
        for i in top_idx if scores[i] > 0
    ]


def rrf(result_lists: list, k: int = 60, top_k: int = 5) -> list[dict]:
    scores, texts = {}, {}
    for results in result_lists:
        for rank, doc in enumerate(results, 1):
            did = doc['id']
            scores[did] = scores.get(did, 0) + 1.0 / (k + rank)
            texts[did]  = doc['text']
    sorted_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [
        {'id': did, 'rrf_score': round(score, 6), 'text': texts[did]}
        for did, score in sorted_docs[:top_k]
    ]


def hybrid_search(
    query: str, collection: str,
    corpus: list, ids: list, bm25_index: BM25Okapi,
    top_k: int = 5
) -> list[dict]:
    sem = semantic_search(query, collection, top_k=20)
    lex = bm25_search(query, corpus, ids, bm25_index, top_k=20)
    return rrf([sem, lex], top_k=top_k)

In [53]:
# ── Commentary retrieval with graceful fallback ───────────

RAW_DIR       = COMMENTARY_DIR / 'raw'
EXTRACTED_DIR = COMMENTARY_DIR / 'extracted'

def get_commentary_context(
    race_name: str,
    race_date: str,
    stage: int = None,
    max_chars: int = 3000
) -> str:
    label     = f"{str(race_date)[:4]} {race_name}"
    label    += f" Stage {stage}" if stage else ""
    safe_name = re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')

    extracted_path = EXTRACTED_DIR / f'{safe_name}.json'
    raw_path       = RAW_DIR       / f'{safe_name}.json'

    # Option 1 — extracted tactical events
    if extracted_path.exists():
        try:
            with open(extracted_path, encoding='utf-8') as f:
                data = json.load(f)
            ext = data.get('extraction', {})
            if 'error' not in ext:
                parts = [
                    f"[COMMENTARY: {data.get('channel')} | "
                    f"{data.get('video_title','')[:55]}]"
                ]
                if ext.get('race_summary'):
                    parts.append(f"Summary: {ext['race_summary']}")
                if ext.get('decisive_moment'):
                    parts.append(f"Decisive moment: {ext['decisive_moment']}")
                for e in ext.get('events', [])[:5]:
                    riders = ', '.join(e.get('riders', []))
                    parts.append(
                        f"- [{e['event_type']}] {e['description']}"
                        + (f" ({riders})" if riders else "")
                    )
                for s in ext.get('rider_form_signals', [])[:3]:
                    parts.append(
                        f"- Form: {s['rider']} ({s['signal']}): {s['observation']}"
                    )
                return '\n'.join(parts)
        except Exception:
            pass

    # Option 2 — raw transcript
    if raw_path.exists():
        try:
            with open(raw_path, encoding='utf-8') as f:
                data = json.load(f)
            if data.get('status') == 'transcript_saved':
                text = data.get('transcript', {}).get('clean_text', '')
                if text:
                    if len(text) > max_chars:
                        half = max_chars // 2
                        text = text[:half] + '\n[...]\n' + text[-half:]
                    channel = data.get('video', {}).get('channel', 'unknown')
                    title   = data.get('video', {}).get('title', '')[:55]
                    return f'[RAW COMMENTARY: {channel} | {title}]\n\n{text}'
        except Exception:
            pass

    # Option 3 — no commentary available
    return f'[NO COMMENTARY AVAILABLE for {label}] Analysis based on structured data only.'

In [54]:
# ── ML prediction function ────────────────────────────────

def predict_stage_context(
    race_name: str,
    year: int,
    stage: int = None,
    top_n: int = 10
) -> str:
    mask = (
        model_df["Race_results"].str.contains(   
            race_name, na=False, case=False
        ) &
        (model_df["Year_results"] == year)        
    )
    if stage is not None:
        mask &= (
            (model_df["Stage_results"] == stage) |
            (model_df["Stage_results"] == float(stage))
        )

    race_rows = model_df[mask].drop_duplicates("Name")  

    if race_rows.empty:
        return f"[NO PREDICTION DATA] No data found for {race_name} {year}"

    missing = [f for f in FEATURE_COLS if f not in race_rows.columns]
    if missing:
        return f"[PREDICTION ERROR] Missing features: {missing[:5]}"

    X = race_rows[FEATURE_COLS].copy()
    if needs_imp:
        X = X.fillna(xgb_medians)

    proba = pred_model.predict_proba(X)

    results_df = race_rows[["Name", "Team"]].copy()
    for i, tier in enumerate(TIER_ORDER):
        results_df[f"p_{tier}"] = proba[:, i]

    top_predictions = (
        results_df
        .sort_values("p_winner", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    label = f"{year} {race_name}" + (f" Stage {stage}" if stage else "")
    lines = [f"[ML PREDICTIONS: {label} | Model: {pred_model_name}]"]
    lines.append(f"Top {top_n} riders by win probability:")
    lines.append("")

    for _, row in top_predictions.iterrows():
        lines.append(
            f"  {row['Name']:<30} "
            f"Win:{row['p_winner']*100:5.1f}% "
            f"Pod:{row['p_podium']*100:5.1f}% "
            f"T10:{row['p_top10']*100:5.1f}%"
        )

    return "\n".join(lines)


# Test
print(predict_stage_context('Tour de France', 2023, stage=17))

[ML PREDICTIONS: 2023 Tour de France Stage 17 | Model: XGBoost]
Top 10 riders by win probability:

  VINGEGAARD Jonas               Win: 11.4% Pod: 20.4% T10: 42.8%
  POGAČAR Tadej                  Win:  9.6% Pod: 16.6% T10: 49.2%
  PIDCOCK Thomas                 Win:  6.1% Pod:  6.9% T10: 16.8%
  GALL Felix                     Win:  5.7% Pod:  5.2% T10: 29.5%
  VAN AERT Wout                  Win:  5.4% Pod:  6.1% T10: 21.9%
  PINOT Thibaut                  Win:  4.0% Pod: 13.8% T10: 38.4%
  YATES Adam                     Win:  4.0% Pod: 13.8% T10: 51.3%
  GAUDU David                    Win:  4.0% Pod: 10.4% T10: 49.3%
  ALAPHILIPPE Julian             Win:  3.4% Pod:  2.1% T10:  7.6%
  BILBAO Pello                   Win:  3.2% Pod: 13.1% T10: 38.2%


In [55]:
# ── Router node ───────────────────────────────────────────

ROUTER_SYSTEM = """You are a query router for PelotonIQ, a professional cycling intelligence system.

Classify the query into exactly one type:

STRUCTURED — asks for a specific fact retrievable directly from a database.
  Examples: who won X race, what place did rider finish, top 10 of X race,
  which riders perform best on mountain stages, best climbers historically
  NOTE: questions asking which riders perform best on specific terrain types
  should use get_best_mountain_riders function, not semantic search.
  
  Functions available:
  - get_stage_winner
  - get_race_results  
  - get_rider_results
  - get_stage_profile
  - get_best_mountain_riders  ← use for "best riders on terrain type X" queries

SEMANTIC_COURSE — asks about race/stage terrain, course profile, surface type, elevation.
  Examples: what makes Paris-Roubaix hard, hardest mountain stages, flat sprint stages,
  which stages have the most climbing, most brutal terrain

SEMANTIC_RIDER — asks about a rider's general performance, season, strengths.
  Examples: how did Pogacar perform in 2023, best climbers, Evenepoel season summary

PREDICTIVE — forward-looking analysis requiring ML model predictions.
  Examples: who should I watch on stage 17, pre-race analysis, stage preview

HYBRID — requires both terrain context AND rider performance to answer.
  Examples: who performs best on cobbled stages, climbers suited to this terrain

Return JSON only:
{
  "query_type": "STRUCTURED|SEMANTIC_COURSE|SEMANTIC_RIDER|PREDICTIVE|HYBRID",
  "reasoning": "one sentence",
  "structured_params": {
    "function": "get_stage_winner|get_race_results|get_rider_results|get_stage_profile",
    "race_name": "...",
    "year": 2023,
    "stage": null,
    "rider_name": "...",
    "top_n": 10
  },
  "race_context": {
    "race_name": "...",
    "year": 2023,
    "stage": null,
    "race_date": "YYYY-MM-DD or null"
  }
}
Only include structured_params if query_type is STRUCTURED.
Always include race_context if a specific race is mentioned."""


def router_node(state: PelotonState) -> PelotonState:
    """Classify query type and extract structured parameters."""
    state['steps_taken'].append('router')

    response = claude.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=800,  # increase from 500
        system=ROUTER_SYSTEM,
        messages=[{'role': 'user', 'content': state['query']}]
    )

    raw = response.content[0].text.strip()
    raw = re.sub(r'```json|```', '', raw).strip()

    try:
        parsed = json.loads(raw)
        state['query_type']       = parsed.get('query_type', 'HYBRID')
        state['structured_params'] = parsed.get('structured_params', {})
        # Store race context for use by other nodes
        state['structured_params']['race_context'] = parsed.get('race_context', {})
    except json.JSONDecodeError:
        state['query_type'] = 'HYBRID'
        state['error'] = f'Router parse error: {raw[:100]}'

    return state

In [56]:
# ── Structured tool node ──────────────────────────────────

def structured_node(state: PelotonState) -> PelotonState:
    """Execute dataframe tool call for structured queries."""
    state['steps_taken'].append('structured_tool')

    params = state.get('structured_params', {})
    result = dispatch_tool(params)
    state['structured_data'] = result

    return state

In [57]:
# ── Retriever node ────────────────────────────────────────

def retriever_node(state: PelotonState) -> PelotonState:
    """Run hybrid search over course and/or rider collections.
    For numeric terrain queries, uses structured Qdrant filters
    instead of semantic search for more precise results.
    """
    state['steps_taken'].append('retriever')
    qt          = state['query_type']
    query_lower = state['query'].lower()

    # Keywords that signal a numeric elevation/terrain filter
    # is more appropriate than semantic search
    elevation_keywords = [
        '2000m', 'above 2000', 'high altitude', 'highest elevation',
        'most climbing', 'most vertical', 'hardest climb',
        'steepest', 'highest point'
    ]
    use_elevation_filter = any(kw in query_lower for kw in elevation_keywords)

    if qt in ['SEMANTIC_COURSE', 'PREDICTIVE', 'HYBRID']:
        if use_elevation_filter:
            # Use Qdrant payload filter for precise elevation queries
            # vertical_gain >= 4000m is a reliable proxy for high mountain stages
            filtered, _ = qdrant.scroll(
                collection_name='course_profiles',
                scroll_filter=Filter(
                    must=[
                        FieldCondition(
                            key='vertical_gain',
                            range=Range(gte=4000)
                        )
                    ]
                ),
                limit=5,
                with_payload=True
            )
            if filtered:
                state['course_context'] = '\n'.join(
                    f"[{r.payload.get('doc_id')}]\n"
                    f"{r.payload.get('text', '')[:400]}"
                    for r in filtered
                )
            else:
                # Fall back to semantic search if filter returns nothing
                results = hybrid_search(
                    state['query'], 'course_profiles',
                    course_corpus, course_ids, bm25_course, top_k=3
                )
                state['course_context'] = '\n'.join(
                    f"[{r['id']}]\n{r['text'][:400]}"
                    for r in results
                )
        else:
            # Standard hybrid search
            results = hybrid_search(
                state['query'], 'course_profiles',
                course_corpus, course_ids, bm25_course, top_k=3
            )
            state['course_context'] = '\n'.join(
                f"[{r['id']}]\n{r['text'][:400]}"
                for r in results
            )

    if qt in ['SEMANTIC_RIDER', 'PREDICTIVE', 'HYBRID']:
        results = hybrid_search(
            state['query'], 'rider_seasons',
            rider_corpus, rider_ids, bm25_rider, top_k=3
        )
        state['rider_context'] = '\n'.join(
            f"[{r['id']}]\n{r['text'][:400]}"
            for r in results
        )

    return state

In [58]:
# ── Commentary node ───────────────────────────────────────

def commentary_node(state: PelotonState) -> PelotonState:
    """Add race commentary context if available."""
    state['steps_taken'].append('commentary')

    # Guard against None race_context
    structured_params = state.get('structured_params') or {}
    race_ctx  = structured_params.get('race_context') or {}
    race_name = race_ctx.get('race_name')
    race_date = race_ctx.get('race_date')
    stage     = race_ctx.get('stage')

    if race_name and race_date:
        state['commentary_context'] = get_commentary_context(
            race_name, race_date, stage
        )
    else:
        state['commentary_context'] = (
            '[NO COMMENTARY] No specific race identified in query.'
        )

    return state

In [59]:
# ── Predictor node ────────────────────────────────────────

def predictor_node(state: PelotonState) -> PelotonState:
    """Run ML model prediction for predictive/hybrid queries."""
    state['steps_taken'].append('predictor')

    # Guard against None race_context
    structured_params = state.get('structured_params') or {}
    race_ctx  = structured_params.get('race_context') or {}
    race_name = race_ctx.get('race_name')
    year      = race_ctx.get('year')
    stage     = race_ctx.get('stage')

    if race_name and year:
        state['prediction_context'] = predict_stage_context(
            race_name, int(year), stage
        )
    else:
        state['prediction_context'] = (
            '[NO PREDICTION] This query asks about general rider performance '
            'across multiple races rather than a specific stage. '
            'Predictions are based on structured race data and rider history.'
        )

    return state

In [60]:
# ── Synthesizer node ──────────────────────────────────────

SYNTHESIZER_SYSTEM = """You are PelotonIQ, a professional cycling intelligence assistant.

You have access to a dataset covering UCI WorldTour races from 2017-2023,
including detailed course profiles, race results, rider performance history,
ML-based finish probability predictions, and race commentary.

Answer using only the provided context. Be specific and cite the data.
For predictions, explain which factors drove the model's output.
If context is missing, say so clearly rather than guessing.
Keep responses concise and analytically focused."""


def build_context_prompt(state: PelotonState) -> str:
    """Assemble all retrieved context into a structured prompt."""
    parts = []

    if state.get('structured_data'):
        parts.append('STRUCTURED DATA LOOKUP:')
        parts.append(json.dumps(state['structured_data'], indent=2, default=str))

    if state.get('course_context'):
        parts.append('\nCOURSE PROFILE CONTEXT:')
        parts.append(state['course_context'])

    if state.get('rider_context'):
        parts.append('\nRIDER PERFORMANCE CONTEXT:')
        parts.append(state['rider_context'])

    if state.get('prediction_context'):
        parts.append('\nML MODEL PREDICTIONS:')
        parts.append(state['prediction_context'])

    if state.get('commentary_context'):
        parts.append('\nRACE COMMENTARY:')
        parts.append(state['commentary_context'])

    return '\n'.join(parts)


def synthesizer_node(state: PelotonState) -> PelotonState:
    """Generate final response using all available context."""
    state['steps_taken'].append('synthesizer')

    context = build_context_prompt(state)

    response = claude.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=1500,
        system=SYNTHESIZER_SYSTEM,
        messages=[{
            'role': 'user',
            'content': f'Context:\n{context}\n\nQuestion: {state["query"]}'
        }]
    )

    state['final_response'] = response.content[0].text
    return state

In [61]:
# ── Conditional routing functions ─────────────────────────

def route_from_router(state: PelotonState) -> str:
    """After router: structured queries go to tool, others to retriever."""
    if state['query_type'] == 'STRUCTURED':
        return 'structured_tool'
    return 'retriever'


def route_from_retriever(state: PelotonState) -> str:
    """After retrieval: predictive queries run the ML model first."""
    if state['query_type'] in ['PREDICTIVE', 'HYBRID']:
        return 'predictor'
    return 'commentary'


def route_from_structured(state: PelotonState) -> str:
    """After structured lookup: go straight to synthesizer."""
    return 'synthesizer'


def route_from_predictor(state: PelotonState) -> str:
    """After prediction: always add commentary then synthesize."""
    return 'commentary'


def route_from_commentary(state: PelotonState) -> str:
    """After commentary: always synthesize."""
    return 'synthesizer'

In [62]:
# ── Build and compile the LangGraph ───────────────────────

graph = StateGraph(PelotonState)

# Add nodes
graph.add_node('router',         router_node)
graph.add_node('structured_tool', structured_node)
graph.add_node('retriever',      retriever_node)
graph.add_node('predictor',      predictor_node)
graph.add_node('commentary',     commentary_node)
graph.add_node('synthesizer',    synthesizer_node)

# Entry point
graph.set_entry_point('router')

# Conditional edges
graph.add_conditional_edges(
    'router',
    route_from_router,
    {
        'structured_tool': 'structured_tool',
        'retriever':        'retriever',
    }
)
graph.add_conditional_edges(
    'retriever',
    route_from_retriever,
    {
        'predictor':   'predictor',
        'commentary':  'commentary',
    }
)
graph.add_conditional_edges(
    'structured_tool',
    route_from_structured,
    {'synthesizer': 'synthesizer'}
)
graph.add_conditional_edges(
    'predictor',
    route_from_predictor,
    {'commentary': 'commentary'}
)
graph.add_conditional_edges(
    'commentary',
    route_from_commentary,
    {'synthesizer': 'synthesizer'}
)

# Terminal edge
graph.add_edge('synthesizer', END)

# Compile
app = graph.compile()
print('Nodes:', list(graph.nodes.keys()))


Nodes: ['router', 'structured_tool', 'retriever', 'predictor', 'commentary', 'synthesizer']


In [63]:
# ── Main query interface ──────────────────────────────────

def ask(query: str, verbose: bool = True) -> str:
    """
    Run a query through the full PelotonIQ agent pipeline.
    Returns the final response string.
    """
    if verbose:
        print(f"\n{'='*65}")
        print(f"Query: {query}")
        print('='*65)

    state  = empty_state(query)
    result = app.invoke(state)

    if verbose:
        print(f"\nSteps: {' → '.join(result['steps_taken'])}")
        print(f"Type:  {result['query_type']}")
        if result.get('error'):
            print(f"Error: {result['error']}")
        print(f"\n{'-'*65}")
        print(result['final_response'])

    return result['final_response']

In [23]:
# ── Showcase queries — run these one at a time ────────────
# Each exercises a different part of the system

# Query 1 — STRUCTURED: direct dataframe lookup
# Expected path: router → structured_tool → synthesizer
_ = ask("Who won Tour de France Stage 17 in 2023?")



Query: Who won Tour de France Stage 17 in 2023?

Steps: router → structured_tool → synthesizer
Type:  STRUCTURED

-----------------------------------------------------------------
# Tour de France 2023 Stage 17 Winner

**Felix GALL** (AG2R Citroën Team) won Stage 17 of the 2023 Tour de France.

This was a significant victory for the Austrian rider, representing AG2R Citroën Team in one of the Tour's mountain stages.


In [24]:
# Query 2 — SEMANTIC_COURSE: terrain analysis
# Expected path: router → retriever → commentary → synthesizer
_ = ask("What makes Paris-Roubaix so different from other classics?")



Query: What makes Paris-Roubaix so different from other classics?

Steps: router → retriever → commentary → synthesizer
Type:  SEMANTIC_COURSE

-----------------------------------------------------------------
Based on the course profile data, Paris-Roubaix stands out for several distinctive characteristics:

## The Cobblestones
**48.7km of cobblestones** (2018 data) - nearly 19% of the race distance is on pavé sectors. This is the defining feature that no other major classic replicates at this scale. The 2018 profile also shows 4.8km of additional unpaved sections.

## Extreme Distance
**256-258km** across the years analyzed - making it one of the longest single-day races on the calendar. This combines with the cobbles to create cumulative fatigue and attrition.

## Minimal Elevation Profile
Despite being classified as "moderately hilly" with ~1,180-1,200m of vertical gain, the race stays relatively flat:
- Maximum elevation: only 161m
- Minimum elevation: 15m
- Vertical range: 146m


In [25]:
# Query 3 — SEMANTIC_RIDER: rider season summary
# Expected path: router → retriever → commentary → synthesizer
_ = ask("How did Remco Evenepoel perform across his seasons in our dataset?")


Query: How did Remco Evenepoel perform across his seasons in our dataset?

Steps: router → retriever → commentary → synthesizer
Type:  SEMANTIC_RIDER

-----------------------------------------------------------------
Based on the dataset covering 2019-2021, Remco Evenepoel's WorldTour performance shows an early-career trajectory with the Deceuninck - Quick Step team:

## Season-by-Season Performance:

**2019** (3 races)
- 0 wins
- 1 top-10 finish
- 1 DNF
- Limited WorldTour participation in debut season

**2020** (2 races)
- 1 win: **Tour de Pologne Stage 4**
- 1 top-10 finish
- 1 DNF
- Breakthrough victory despite shortened season

**2021** (3 races)
- 0 wins
- 3 top-10 finishes (best improvement)
- 2 DNFs
- **Notable**: 4th place at Giro d'Italia (best Grand Tour result in dataset)

## Key Observations:

**Consistency Issues**: High DNF rate (4 DNFs across 8 total races = 50%) suggests either crashes, illness, or tactical withdrawals during this period.

**Progression**: Top-10 fini

In [26]:
# Query 4 — PREDICTIVE: the showcase query
# Expected path: router → retriever → predictor → commentary → synthesizer
_ = ask(
    "It's before stage 17 of the 2023 Tour de France. "
    "Who should I watch and why? Give me a pre-race briefing."
)



Query: It's before stage 17 of the 2023 Tour de France. Who should I watch and why? Give me a pre-race briefing.

Steps: router → retriever → predictor → commentary → synthesizer
Type:  PREDICTIVE

-----------------------------------------------------------------
# Pre-Race Briefing: 2023 Tour de France Stage 17

## The Stage
This is **the queen stage** of the 2023 Tour - the final and most brutal Alpine test. You're looking at:
- **165.8 km** through the Giants of the Alps
- **5,570m of climbing** (massive vertical gain)
- Four categorized climbs culminating in the **Hors Catégorie finish to Courchevel**
- Maximum elevation of **2,299m** - the highest point of this year's race

## Key Riders to Watch

### **1. Jonas Vingegaard (11.4% win probability)**
- The ML model's favorite despite Pogačar's dominant season
- Currently in yellow with a commanding **7:35 lead** over Pogačar
- This stage profile suits his exceptional climbing ability
- Watch for: Whether he can extend his already h

In [ ]:
# Query 5 — HYBRID: terrain + rider context
# Expected path: router → retriever → predictor → commentary → synthesizer  
_ = ask(
    "Which riders in our dataset have historically performed best "
    "on high mountain stages above 2000m elevation?"
)



Query: Which riders in our dataset have historically performed best on high mountain stages above 2000m elevation?

Steps: router → structured_tool → synthesizer
Type:  STRUCTURED

-----------------------------------------------------------------
Based on the dataset of 116 mountain stages with minimum 4,000m vertical gain from 2017-2023, the top performers on high mountain stages are:

## Elite Tier (>65% top-10 rate):

**1. Tadej POGAČAR (UAE Team Emirates)**
- **75% top-10 rate** (18/24 races)
- 7 wins, avg rank: 6.4
- Dominant consistency across all mountain terrain

**2. Juan AYUSO (UAE Team Emirates)**  
- **72.7% top-10 rate** (8/11 races)
- 1 win, avg rank: 10.3
- Strong emerging climber, limited sample size

**3. Remco EVENEPOEL (Deceuninck - Quick Step)**
- **68.8% top-10 rate** (11/16 races)
- 5 wins, avg rank: 13.2
- High win conversion on selective mountain stages

**4. Alberto CONTADOR (Trek - Segafredo)**
- **66.7% top-10 rate** (4/6 races)
- 0 wins, avg rank: 10.0
- La

: 

In [22]:
# ── Routing accuracy evaluation ───────────────────────────
# Validate the router is classifying queries correctly
# before running full pipeline on each

routing_tests = [
    ("Who won TDF Stage 19 in 2023?",                      'STRUCTURED'),
    ("What is Paris-Roubaix like terrain-wise?",            'SEMANTIC_COURSE'),
    ("How did Vingegaard perform in 2023?",                 'SEMANTIC_RIDER'),
    ("Give me a pre-race briefing for TDF 2023 Stage 17",  'PREDICTIVE'),
    ("Who performs best on cobbled mountain stages?",       'HYBRID'),
    ("Top 10 results of Strade Bianche 2022",              'STRUCTURED'),
    ("Which stages in our dataset have the most climbing?", 'SEMANTIC_COURSE'),
]

print('=== ROUTING ACCURACY TEST ===\n')
correct = 0
for query, expected in routing_tests:
    state  = empty_state(query)
    result = router_node(state)
    actual = result['query_type']
    match  = actual == expected
    correct += match
    icon   = '✓' if match else '✗'
    print(f"{icon} {query[:55]:<55}")
    print(f"  Expected: {expected:<20} Got: {actual}")
    print()

print(f"Routing accuracy: {correct}/{len(routing_tests)} "
      f"({correct/len(routing_tests)*100:.0f}%)")


=== ROUTING ACCURACY TEST ===

✓ Who won TDF Stage 19 in 2023?                          
  Expected: STRUCTURED           Got: STRUCTURED

✓ What is Paris-Roubaix like terrain-wise?               
  Expected: SEMANTIC_COURSE      Got: SEMANTIC_COURSE

✓ How did Vingegaard perform in 2023?                    
  Expected: SEMANTIC_RIDER       Got: SEMANTIC_RIDER

✓ Give me a pre-race briefing for TDF 2023 Stage 17      
  Expected: PREDICTIVE           Got: PREDICTIVE

✓ Who performs best on cobbled mountain stages?          
  Expected: HYBRID               Got: HYBRID

✓ Top 10 results of Strade Bianche 2022                  
  Expected: STRUCTURED           Got: STRUCTURED

✓ Which stages in our dataset have the most climbing?    
  Expected: SEMANTIC_COURSE      Got: SEMANTIC_COURSE

Routing accuracy: 7/7 (100%)


In [19]:
# ── Free sanity check — no API calls ──────────────────────

print("=== SYSTEM SANITY CHECK ===\n")

# 1. Data loaded
print(f"✓ merged_df: {merged_df.shape[0]:,} rows") if len(merged_df) > 0 else print("✗ merged_df empty")

# 2. Model loaded
print(f"✓ Model: {pred_model_name}") if pred_model else print("✗ No model loaded")

# 3. Qdrant collections
collections = [c.name for c in qdrant.get_collections().collections]
for expected in ["course_profiles", "rider_seasons"]:
    if expected in collections:
        info = qdrant.get_collection(expected)
        print(f"✓ Qdrant '{expected}': {info.points_count:,} points")
    else:
        print(f"✗ Qdrant '{expected}': MISSING — re-run notebook 2")

# 4. BM25 indexes
print(f"✓ BM25 course: {len(course_corpus):,} docs") if course_corpus else print("✗ BM25 course empty")
print(f"✓ BM25 rider:  {len(rider_corpus):,} docs") if rider_corpus else print("✗ BM25 rider empty")

# 5. Commentary directory
raw_count = len(list(RAW_DIR.glob("*.json"))) if RAW_DIR.exists() else 0
print(f"✓ Commentary: {raw_count} raw transcripts available")

# 6. Tool functions work
winner = get_stage_winner("Tour de France", 2023, stage=17)
if "error" not in winner:
    print(f"✓ Tool: get_stage_winner → {winner['winner']}")
else:
    print(f"✗ Tool: {winner['error']}")

# 7. Prediction works
pred = predict_stage_context("Tour de France", 2023, stage=17)
if "NO PREDICTION" not in pred and "ERROR" not in pred:
    print(f"✓ Prediction: returned {len(pred)} chars")
else:
    print(f"✗ Prediction: {pred[:80]}")

# 8. Retrieval works
results = hybrid_search(
    "mountain stage climbing",
    "course_profiles",
    course_corpus, course_ids, bm25_course,
    top_k=1
)
print(f"✓ Retrieval: top result = {results[0]['id']}") if results else print("✗ Retrieval: no results")

# 9. Commentary works
ctx = get_commentary_context("Tour de France", "2023-07-19", 17)
print(f"✓ Commentary: {ctx[:60]}...")

print("\n=== ALL CHECKS PASSED — safe to run Cell 23 ===")

=== SYSTEM SANITY CHECK ===

✓ merged_df: 140,573 rows
✓ Model: XGBoost
✓ Qdrant 'course_profiles': 963 points
✓ Qdrant 'rider_seasons': 5,835 points
✓ BM25 course: 963 docs
✓ BM25 rider:  5,835 docs
✓ Commentary: 63 raw transcripts available
✓ Tool: get_stage_winner → GALL Felix
✓ Prediction: returned 758 chars
✓ Retrieval: top result = 2019 Paris-Nice Stage 7
✓ Commentary: [RAW COMMENTARY: NBC Sports | Tour de France 2023: Stage 17 ...

=== ALL CHECKS PASSED — safe to run Cell 23 ===
